<a href="https://colab.research.google.com/github/adaleidman/Data_Science_Project_AL/blob/main/DataCollectionAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install libraries

!pip install yfinance pandas numpy

In [2]:
# Import

import yfinance as yf
import pandas as pd
import numpy as np
import os

In [3]:
# Download the S&P 500 data

end = pd.Timestamp.today().normalize()
start = end - pd.DateOffset(years=15)

df = yf.download("^GSPC", start=start, end=end, auto_adjust=True, progress=False)

# flatten columns if needed, keep the five we want
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)
df = df.rename(columns=str.title)[["Open", "High", "Low", "Close", "Volume"]]
df.index.name = "Date"

print("Downloaded", len(df), "rows, from", df.index.min().date(), "to", df.index.max().date())
df.head()

Downloaded 3771 rows, from 2011-06-15 to 2026-06-12


Price,Open,High,Low,Close,Volume
Date,,,,,
2011-06-15,1287.869995,1287.869995,1261.900024,1265.420044,4070500000
2011-06-16,1265.530029,1274.109985,1258.069946,1267.640015,3846250000
2011-06-17,1268.579956,1279.819946,1267.400024,1271.500000,4916460000
2011-06-20,1271.500000,1280.420044,1267.560059,1278.359985,3464660000
2011-06-21,1278.400024,1297.619995,1278.400024,1295.520020,4056150000


In [4]:
# Save the raw data

os.makedirs("data/raw", exist_ok=True)
df.to_csv("data/raw/sp500_raw.csv")
print("Saved data/raw/sp500_raw.csv")

Saved data/raw/sp500_raw.csv


In [6]:
# Returns

df["return"] = df["Close"].pct_change()
df["log_return"] = np.log(df["Close"] / df["Close"].shift(1))

In [7]:
# Moving averages (trend)

for w in (10, 20, 50):
    df[f"sma_{w}"] = df["Close"].rolling(w).mean()
df["ema_12"] = df["Close"].ewm(span=12, adjust=False).mean()
df["ema_26"] = df["Close"].ewm(span=26, adjust=False).mean()


In [8]:
# RSI (momentum)

delta = df["Close"].diff()
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)
avg_gain = gain.ewm(alpha=1/14, min_periods=14, adjust=False).mean()
avg_loss = loss.ewm(alpha=1/14, min_periods=14, adjust=False).mean()
df["rsi_14"] = 100 - 100 / (1 + avg_gain / avg_loss)

In [9]:
# MACD (trend/momentum)

df["macd"] = df["ema_12"] - df["ema_26"]
df["macd_signal"] = df["macd"].ewm(span=9, adjust=False).mean()
df["macd_hist"] = df["macd"] - df["macd_signal"]

In [10]:
# Bollinger Bands (volatility)

mid = df["Close"].rolling(20).mean()
std = df["Close"].rolling(20).std()
df["bb_upper"] = mid + 2 * std
df["bb_lower"] = mid - 2 * std
df["bb_width"] = (df["bb_upper"] - df["bb_lower"]) / mid

In [11]:
# ATR and momentum

prev_close = df["Close"].shift(1)
tr = pd.concat([df["High"] - df["Low"],
                (df["High"] - prev_close).abs(),
                (df["Low"] - prev_close).abs()], axis=1).max(axis=1)
df["atr_14"] = tr.ewm(alpha=1/14, min_periods=14, adjust=False).mean()
df["momentum_10"] = df["Close"] - df["Close"].shift(10)

In [12]:
# The prediction targets

df["next_return"] = df["Close"].pct_change().shift(-1)          # tomorrow's return
df["target_up"] = (df["Close"].shift(-1) > df["Close"]).astype(int)  # 1 = up, 0 = down

In [13]:
# Clean up and check

df = df.dropna()   # remove warm-up rows and the last row (no future target)
print("Final shape:", df.shape)
print("Up days:", round(df["target_up"].mean() * 100, 1), "%")
df.head()

Final shape: (3721, 23)
Up days: 54.6 %


Price,Open,High,Low,Close,Volume,return,log_return,sma_10,sma_20,sma_50,...,macd,macd_signal,macd_hist,bb_upper,bb_lower,bb_width,atr_14,momentum_10,next_return,target_up
Date,,,,,,,,,,,,,,,,,,,,,
2011-08-24,1162.160034,1178.560059,1156.300049,1177.599976,5315310000,0.013120,0.013035,1167.054004,1193.851001,1264.202202,...,-35.098834,-35.430931,0.332097,1309.868828,1077.833174,0.194359,33.402270,56.839966,-0.015566,0
2011-08-25,1176.689941,1190.680054,1155.469971,1159.270020,5748420000,-0.015566,-0.015688,1165.717004,1186.781000,1262.079202,...,-33.537487,-35.052242,1.514756,1292.134025,1081.427974,0.177544,33.531400,-13.369995,0.015122,1
2011-08-26,1158.849976,1181.229980,1135.910034,1176.800049,5035320000,0.015122,0.015008,1165.516003,1181.007001,1260.262402,...,-30.533609,-34.148516,3.614907,1273.940795,1088.073207,0.157381,34.373439,-2.010010,0.028280,1
2011-08-29,1177.910034,1210.280029,1177.910034,1210.079956,4228070000,0.028280,0.027888,1166.075000,1177.164001,1259.034001,...,-25.177374,-32.354287,7.176913,1257.101284,1097.226719,0.135813,34.309620,5.589966,0.002347,1
2011-08-30,1209.760010,1220.099976,1195.770020,1212.920044,4572570000,0.002347,0.002344,1168.091003,1175.107501,1257.725203,...,-20.467413,-29.976912,9.509500,1248.570417,1101.644585,0.125032,33.596787,20.160034,0.004922,1


In [14]:
# Save the finished dataset

os.makedirs("data/processed", exist_ok=True)
df.to_csv("data/processed/sp500_features.csv")
print("Saved data/processed/sp500_features.csv")

Saved data/processed/sp500_features.csv
